# Episode V — From One Neuron to a Brain

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Pchambet/Deep-Learning-from-Scratch/blob/main/notebooks/05_from_neuron_to_brain.ipynb)

**Goal**: Build and train a two-layer neural network from scratch that can solve nonlinear classification problems.

## Why Do We Need Multiple Layers?

In previous episodes, we saw how a **single neuron** (perceptron) can classify linearly separable data — think of a straight line (in 2D) or a hyperplane (in higher dimensions) dividing two classes.

But what happens when the data is **not linearly separable**? Consider the classic **XOR problem** or **concentric circles** — no single straight line can separate the classes.

This is where **neural networks** come in. By stacking neurons in layers and introducing **nonlinear activation functions**, we can learn arbitrarily complex decision boundaries.

## Architecture: The Two-Layer Network

Our network has:
- **Input layer**: $n_0$ features (e.g., pixel values, coordinates)
- **Hidden layer**: $n_1$ neurons with **tanh** activation (introduces nonlinearity)
- **Output layer**: $n_2 = 1$ neuron with **sigmoid** activation (binary classification)

### Mathematical Formulation

**Forward Propagation:**

$$
Z^{[1]} = W^{[1]} X + b^{[1]} \quad \text{(hidden layer pre-activation)}
$$

$$
A^{[1]} = \tanh(Z^{[1]}) \quad \text{(hidden layer activation)}
$$

$$
Z^{[2]} = W^{[2]} A^{[1]} + b^{[2]} \quad \text{(output layer pre-activation)}
$$

$$
A^{[2]} = \sigma(Z^{[2]}) \quad \text{(output probabilities)}
$$

**Loss Function (Binary Cross-Entropy):**

$$
\mathcal{L} = -\frac{1}{m} \sum_{i=1}^{m} \left[ y^{(i)} \log(a^{[2](i)}) + (1 - y^{(i)}) \log(1 - a^{[2](i)}) \right]
$$

**Backward Propagation:**

Using the chain rule, we compute gradients:

$$
dZ^{[2]} = A^{[2]} - Y
$$

$$
dW^{[2]} = \frac{1}{m} dZ^{[2]} (A^{[1]})^T
$$

$$
db^{[2]} = \frac{1}{m} \sum dZ^{[2]}
$$

$$
dZ^{[1]} = (W^{[2]})^T dZ^{[2]} \odot (1 - (A^{[1]})^2) \quad \text{(tanh derivative)}
$$

$$
dW^{[1]} = \frac{1}{m} dZ^{[1]} X^T
$$

$$
db^{[1]} = \frac{1}{m} \sum dZ^{[1]}
$$

**Weight Update:**

$$
W^{[l]} := W^{[l]} - \alpha \, dW^{[l]}
$$

$$
b^{[l]} := b^{[l]} - \alpha \, db^{[l]}
$$

where $\alpha$ is the learning rate.

---

Now let's see this in action!

## Setup

Import required libraries and our custom two-layer network module.

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_circles

_root = os.getcwd() if os.path.isdir(os.path.join(os.getcwd(), 'src')) else os.path.dirname(os.getcwd())
sys.path.insert(0, _root)

from src.two_layer_network import fit_two_layer_network, predict

## Part 1: Concentric Circles (Nonlinear Separation)

We'll use scikit-learn's `make_circles` to generate a classic **nonlinearly separable dataset**:
- Inner circle: one class
- Outer ring: another class

A single neuron (linear classifier) would fail spectacularly here. But our two-layer network should learn to **wrap around** the inner circle.

### Hyperparameters
- **n1 = 32**: number of hidden neurons (more neurons → more expressive power)
- **learning_rate = 0.2**: step size for gradient descent
- **epochs = 3000**: training iterations
- **seed = 42**: reproducibility

In [ ]:
# Generate concentric circles dataset
X_raw, y_raw = make_circles(n_samples=400, noise=0.08, factor=0.35, random_state=42)
X = X_raw.T  # Shape: (2, 400) — two features (x, y coordinates)
y = y_raw.reshape(1, -1)  # Shape: (1, 400) — binary labels

print(f"Dataset shape: X = {X.shape}, y = {y.shape}")
print(f"Class distribution: {np.sum(y == 0)} circles (inner), {np.sum(y == 1)} circles (outer)")

# Train the two-layer network
run = fit_two_layer_network(X, y, n1=32, learning_rate=0.2, epochs=3000, seed=42)

print(f"\n✅ Training complete!")
print(f"Final accuracy: {run['accuracy'][-1]:.4f}")
print(f"Final loss: {run['loss'][-1]:.6f}")

### Training Curves

We expect:
- **Loss** to decrease monotonically (network is learning)
- **Accuracy** to approach 100% (perfect separation)

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(run['loss'], linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Binary Cross-Entropy Loss')
plt.title('Loss Curve')
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(run['accuracy'], linewidth=2, color='green')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy Curve')
plt.ylim([0.4, 1.05])
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Decision Boundary Visualization

The network learned a **circular decision boundary** separating the two classes. This is impossible for a single neuron!

The colored regions show the model's predicted class for each point in the 2D plane:
- **Blue region**: predicted class 0 (inner circle)
- **Red region**: predicted class 1 (outer ring)

In [ ]:
# Create a mesh grid to visualize decision boundary
x_min, x_max = X[0, :].min() - 1, X[0, :].max() + 1
y_min, y_max = X[1, :].min() - 1, X[1, :].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.01), np.arange(y_min, y_max, 0.01))
grid = np.c_[xx.ravel(), yy.ravel()].T
Z = predict(grid, run['parameters']).reshape(xx.shape)

plt.figure(figsize=(8, 8))
plt.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.Spectral)
plt.scatter(X[0, :], X[1, :], c=y.flatten(), s=50, edgecolors='k', cmap=plt.cm.Spectral)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Episode V: Two-Layer Network Decision Boundary\n(Concentric Circles)', fontsize=14)
plt.colorbar(label='Predicted Class')
plt.show()

---

## Part 2: Cats vs Dogs Image Classification

Now let's tackle a **real-world problem**: classifying images of cats and dogs.

### Dataset
- **Training set**: 1000 images (64×64 pixels, RGB)
- **Test set**: 200 images
- **Labels**: 0 = cat, 1 = dog

Each image has $64 \times 64 \times 3 = 12,288$ features (pixel values). We'll flatten them into a 1D vector and feed them to our two-layer network.

### Data Loading Strategy
- **Local execution**: Load from `data/trainset.hdf5` and `data/testset.hdf5`
- **Google Colab**: Attempt to download from GitHub Releases, or fallback to synthetic demo data

In [ ]:
import importlib.util

# Environment detection
IN_COLAB = 'google.colab' in sys.modules
REQUIRED = ["h5py"]

def _is_installed(pkg: str) -> bool:
    return importlib.util.find_spec(pkg) is not None

missing = [p for p in REQUIRED if not _is_installed(p)]
if missing:
    print("Installing:", ", ".join(missing))
    if IN_COLAB:
        get_ipython().run_line_magic("pip", "install -q " + " ".join(missing))
    else:
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("✅ h5py already installed.")

# Find repository root
def _find_root():
    cwd = os.path.abspath(os.getcwd())
    for _ in range(5):
        if os.path.isdir(os.path.join(cwd, "src")) and os.path.isdir(os.path.join(cwd, "data")):
            return cwd
        parent = os.path.dirname(cwd)
        if parent == cwd:
            break
        cwd = parent
    return None

_root = _find_root()

# On Colab: clone repository if needed
if _root is None or not os.path.isdir(os.path.join(_root, "src")):
    if IN_COLAB:
        import subprocess
        _root = "/content/Deep-Learning-from-Scratch"
        if not os.path.isdir(_root):
            print("Cloning repository...")
            subprocess.run(["git", "clone", "--depth", "1",
                "https://github.com/Pchambet/Deep-Learning-from-Scratch.git", _root],
                check=True, capture_output=True)
    else:
        _root = os.getcwd()

os.chdir(_root)
sys.path.insert(0, _root)

# Ensure data files exist (download or generate synthetic)
def _ensure_data():
    train_path = os.path.join(_root, "data", "trainset.hdf5")
    if os.path.isfile(train_path):
        print("✅ Data files found.")
        return
    data_dir = os.path.join(_root, "data")
    os.makedirs(data_dir, exist_ok=True)
    if IN_COLAB:
        try:
            from urllib.request import urlretrieve
            base = "https://github.com/Pchambet/Deep-Learning-from-Scratch/releases/download/v0.1-data"
            print("Attempting to download data from GitHub Releases...")
            urlretrieve(f"{base}/trainset.hdf5", train_path)
            urlretrieve(f"{base}/testset.hdf5", os.path.join(data_dir, "testset.hdf5"))
            print("✅ Data downloaded from GitHub Releases.")
            return
        except Exception:
            print("⚠️ Download failed, generating synthetic demo data...")
    import h5py
    rng = np.random.default_rng(42)
    for name, n, xk, yk in [
        ("trainset.hdf5", 1000, "X_train", "Y_train"),
        ("testset.hdf5", 200, "X_test", "Y_test")
    ]:
        X_img = rng.integers(0, 256, (n, 64, 64), dtype=np.uint8)
        y_lbl = (rng.random((n, 1)) > 0.5).astype(np.float64)
        with h5py.File(os.path.join(data_dir, name), "w") as f:
            f.create_dataset(xk, data=X_img)
            f.create_dataset(yk, data=y_lbl)
    print("✅ Synthetic demo data created (same format as real data).")
    print("   For real cat/dog images, see data/README_DATA.md")

_ensure_data()

print("\n✅ Environment ready for cats vs dogs classification.")

### Load and Preprocess Data

We'll:
1. Load raw images from HDF5 files
2. Flatten each image from (64, 64) to a 1D vector (12,288 features)
3. Normalize pixel values to [0, 1] (helps gradient descent)
4. Transpose to get shape (features, samples) for our network

In [ ]:
import h5py

def load_cats_dogs_data():
    """Load and preprocess cats vs dogs dataset."""
    data_dir = os.path.join(_root, "data")
    
    with h5py.File(os.path.join(data_dir, "trainset.hdf5"), "r") as f:
        X_train_raw = np.array(f["X_train"])
        y_train_raw = np.array(f["Y_train"])
    
    with h5py.File(os.path.join(data_dir, "testset.hdf5"), "r") as f:
        X_test_raw = np.array(f["X_test"])
        y_test_raw = np.array(f["Y_test"])
    
    # Flatten and normalize
    X_train_flat = X_train_raw.reshape(X_train_raw.shape[0], -1).T / 255.0
    X_test_flat = X_test_raw.reshape(X_test_raw.shape[0], -1).T / 255.0
    
    y_train = y_train_raw.reshape(1, -1)
    y_test = y_test_raw.reshape(1, -1)
    
    return X_train_flat, y_train, X_test_flat, y_test, X_train_raw, X_test_raw

X_train, y_train, X_test, y_test, X_train_img, X_test_img = load_cats_dogs_data()

print(f"Training set: X_train = {X_train.shape}, y_train = {y_train.shape}")
print(f"Test set: X_test = {X_test.shape}, y_test = {y_test.shape}")
print(f"Number of features: {X_train.shape[0]}")
print(f"Class distribution (train): {np.sum(y_train == 0)} cats, {np.sum(y_train == 1)} dogs")
print(f"Class distribution (test): {np.sum(y_test == 0)} cats, {np.sum(y_test == 1)} dogs")

### Visualize Sample Images

Let's look at a few training examples to understand the data.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
axes = axes.flatten()

for i in range(10):
    axes[i].imshow(X_train_img[i], cmap='gray')
    label = 'Dog' if y_train[0, i] == 1 else 'Cat'
    axes[i].set_title(f'{label}')
    axes[i].axis('off')

plt.suptitle('Sample Training Images (Cats vs Dogs)', fontsize=14)
plt.tight_layout()
plt.show()

### Train Two-Layer Network on Cats vs Dogs

We'll use the same architecture as before, but with:
- **n0 = 12,288**: input features (64×64 grayscale pixels)
- **n1 = 64**: hidden neurons (tuned for image classification)
- **learning_rate = 0.01**: lower learning rate for stability with high-dimensional data
- **epochs = 2000**: more iterations to learn complex patterns

**Note**: This will take a few minutes to train. Watch the loss decrease!

In [ ]:
print("Training two-layer network on cats vs dogs...\n")

cats_run = fit_two_layer_network(
    X_train, y_train,
    n1=64,
    learning_rate=0.01,
    epochs=2000,
    seed=42
)

# Evaluate on training set
train_preds = predict(X_train, cats_run['parameters'])
from sklearn.metrics import accuracy_score
train_acc = accuracy_score(y_train.flatten(), train_preds.flatten())

# Evaluate on test set
test_preds = predict(X_test, cats_run['parameters'])
test_acc = accuracy_score(y_test.flatten(), test_preds.flatten())

print(f"\n✅ Training complete!")
print(f"Final training loss: {cats_run['loss'][-1]:.6f}")
print(f"Training accuracy: {train_acc:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

### Training Curves for Cats vs Dogs

Observe how the network learns over time. Image classification is harder than geometric patterns, so expect:
- Slower convergence
- Some fluctuations in accuracy
- Final test accuracy around 65-75% (respectable for a shallow network on raw pixels!)

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(cats_run['loss'], linewidth=2, color='darkred')
plt.xlabel('Epoch')
plt.ylabel('Binary Cross-Entropy Loss')
plt.title('Loss Curve (Cats vs Dogs)')
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(cats_run['accuracy'], linewidth=2, color='darkgreen')
plt.axhline(train_acc, color='green', linestyle='--', label=f'Train acc: {train_acc:.3f}')
plt.axhline(test_acc, color='blue', linestyle='--', label=f'Test acc: {test_acc:.3f}')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy Curve (Cats vs Dogs)')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Visualize Predictions

Let's see some test set predictions to understand where the model succeeds and fails.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
axes = axes.flatten()

for i in range(10):
    idx = i * 20  # Sample every 20th image
    axes[i].imshow(X_test_img[idx], cmap='gray')
    
    true_label = 'Dog' if y_test[0, idx] == 1 else 'Cat'
    pred_label = 'Dog' if test_preds[0, idx] == 1 else 'Cat'
    color = 'green' if true_label == pred_label else 'red'
    
    axes[i].set_title(f'True: {true_label}\nPred: {pred_label}', color=color, fontsize=10)
    axes[i].axis('off')

plt.suptitle('Test Set Predictions (Green = Correct, Red = Incorrect)', fontsize=14)
plt.tight_layout()
plt.show()

---

## Conclusion

In this episode, we built a **two-layer neural network from scratch** and demonstrated its power on two problems:

1. **Concentric circles**: Perfect separation (100% accuracy) — impossible for a single neuron
2. **Cats vs Dogs**: Real-world image classification with respectable performance (~65-75% test accuracy)

### Key Takeaways

✅ **Nonlinear activation functions** (tanh) enable networks to learn complex patterns  
✅ **Backpropagation** efficiently computes gradients for all parameters  
✅ **Hidden layers** act as feature extractors, transforming raw inputs into useful representations  
✅ **Shallow networks** (2 layers) can already solve nontrivial problems  

### What's Next?

Our two-layer network is powerful, but still limited:
- Performance ceiling on complex image tasks
- No spatial structure awareness (treats pixels independently)

In the next episodes, we'll explore:
- **Deeper networks** (3+ layers) for hierarchical feature learning
- **Convolutional layers** for spatial pattern recognition
- **Regularization** to prevent overfitting

---

**Exercise**: Try different hyperparameters (n1, learning rate, epochs) and observe their impact on training curves and test accuracy!